# Step 3: assessment of the val runs → single test run
Reads the 6 runs saved by `02_siamese_training.ipynb` (`checkpoints/`); nothing is trained here. Shared code comes from `model.py` and `training.py`. The sections are the questions, in order: each one depends on the one before. Analysis cells are David's.

| § | Question | Outcome |
|---|---|---|
| 1 | Which pooling goes to test? | Tie within seed noise → **mean, seed 0** |
| 2 | Option B (siamese score + ZIP → LR): needed? | **No**: easy rejection 99.8–99.9% without the ZIP |
| 3 | Why did the 2026-10-21 run break? | **Unshuffled train loader** (David's sampler check) |
| 4 | Single test run, once, after §1–2 are decided → `results.md` | next |

In [1]:
%load_ext autoreload
%autoreload 2

import itertools
import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from helpers.pairs import load_pairs
from helpers.check_model_performance import review_typical_slices

from siamese_model.model import build_vocab, PairDataset, make_collate
from siamese_model.training import (predict, evaluate, broken_run_problems, load_run, rebuild_model,
                                    THRESHOLD, CASE1_GUARDRAIL)

POOLINGS = ["mean", "max"]
SEEDS = [0, 1, 2]
BATCH_SIZE = 64

In [2]:
ot_train, ot_train_label = load_pairs("overture_case2s", "train")   # only for the vocab (train split only, same as 02)
ot_val, ot_val_label = load_pairs("overture_case2s", "val")

vocab = build_vocab(ot_train)
val_loader = DataLoader(PairDataset(ot_val, ot_val_label), batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(vocab))   # not shuffled: predict returns loader order

## 0. Load the runs
`summary`: one row per run (its best epoch's val numbers, epochs run, broken-run problems), from David's summary cell in `02_siamese_training.ipynb` §8. `histories[(pooling, seed)]`: every epoch of that run (`epoch, train_loss, val_loss, found, case1, easy`). Models: `rebuild_model(load_run(seed, pooling)[1])`.

In [3]:
summary_rows, histories = [], {}
for pool_mode, seed in itertools.product(POOLINGS, SEEDS):
    history, ckpt = load_run(seed, pool_mode)
    histories[(pool_mode, seed)] = history
    summary_rows.append({"pooling": pool_mode, "seed": seed, **(ckpt["best"] or {}), "epochs_run": len(history), "problems": broken_run_problems(history)})
summary = pd.DataFrame(summary_rows)
summary

,pooling,seed,epoch,train_loss,val_loss,found,case1,easy,epochs_run,problems
0,mean,0,10,0.003717,0.016915,87.076537,99.434003,99.932951,14,[]
1,mean,1,4,0.010259,0.014307,87.076537,99.499311,99.899426,8,[]
2,mean,2,5,0.008865,0.015387,90.840652,99.201800,99.878473,9,[]
3,max,0,9,0.004218,0.018681,88.958595,99.274363,99.895235,13,[]
4,max,1,6,0.007373,0.015520,88.205772,99.317901,99.886854,10,[]
5,max,2,5,0.008196,0.018080,90.213300,98.955083,99.832376,9,[]


## 1. Pooling choice
**Rule (agreed before computing):** a setting wins only if the gap between the two poolings' means is bigger than the seed spread (max − min over seeds) within each pooling; otherwise it's a tie. Tie on val found → lower val loss (the spec's epoch tie-break, extended); tie on that too → precedent.

In [4]:
# Per pooling: mean over the 3 seeds
pooling_means = summary.groupby("pooling")[["train_loss", "val_loss", "found", "case1", "easy"]].mean()
display(pooling_means)
# Val loss: is the gap between poolings bigger than the seed spread within each pooling?


val_loss_gap = pooling_means.loc["mean", "val_loss"] - pooling_means.loc["max", "val_loss"]
val_loss_seed_spread = summary.groupby("pooling")["val_loss"].agg(lambda s: s.max() - s.min())
print(f"val loss gap (mean pooling − max pooling): {val_loss_gap:.4f}")
print("val loss seed spread (max − min over seeds):")
print(val_loss_seed_spread)

,train_loss,val_loss,found,case1,easy
pooling,,,,,
max,0.006595,0.017427,89.125889,99.182449,99.871489
mean,0.007613,0.015536,88.331242,99.378371,99.903616


val loss gap (mean pooling − max pooling): -0.0019
val loss seed spread (max − min over seeds):
pooling
max     0.003162
mean    0.002608
Name: val_loss, dtype: float64


**Observations (2026-10-23)**
- **Found: tie.** Max averages slightly higher, but the two poolings' seed ranges overlap (max's range sits inside mean's), and max wins only 2 of 3 seeds. Changing only the seed moves found more than changing the pooling does.
- **Val loss: tie.** Mean's lower average is a smaller gap than the seed spread within either pooling, so the seed alone could produce it.
- **Both poolings beat both baselines on val found in every run** (LogReg 81.8, RF 85.4), so neither choice leaves a better-than-baseline model behind.
- **Decision: mean pooling, seed 0.** Mean on precedent (Sentence-BERT, Neculoiu et al. both use mean pooling); seed 0 because it was the default before any run, not picked on val (picking the best-looking seed would send the luckiest run to test).
- **Caveats for the writeup.** (1) Each run's number is its best epoch, and found swings 2–4 pts between neighbouring epochs, so val numbers are slightly optimistic for both poolings. (2) The test number comes from one seed and carries that seed's luck: report it next to the val seed spread. (3) Finding: at 3 seeds, pooling (mean vs max) makes no detectable difference on this data.

## 2. Option B
Spec: put the ZIP back (siamese score + ZIP → LR) only if easy rejection falls behind without it. David's prediction: easy rejection < 98% without the ZIP.

In [5]:
chosen_run = summary[(summary["pooling"] == "mean") & (summary["seed"] == 0)]
chosen_run

,pooling,seed,epoch,train_loss,val_loss,found,case1,easy,epochs_run,problems
0,mean,0,10,0.003717,0.016915,87.076537,99.434003,99.932951,14,[]


**Observations (2026-10-23)**
- **Prediction wrong:** easy rejection is 99.8–99.9% in every run (chosen run 99.93) vs LogReg's 100.0 on val.
- **Why:** easy negatives are random pairs, almost always with a different street name, so the street text alone separates them. The ZIP mattered for LR because LR's street feature was weak (Step 1 street weight 0.6 vs ZIP 16); the siamese model reads the street characters directly.
- LR's 100% is not overfitting: it is measured on held-out val, and comes from the ZIP features (a different ZIP almost guarantees a non-match).
- **Caveat:** some address strings contain town + ZIP text (e.g. the UConn Foursquare string), so the model may not be entirely ZIP-free; the planned ZIP-text diagnostic (open questions) would size this.
- **Decision: no option B.** The gap is a fraction of a point on the easiest slice, while the siamese model is well ahead on found, the hard slice.

## 3. Why did the 2026-10-21 run break?
**Resolved (David, 2026-10-23):** an unshuffled train loader. David's own sampler check confirms the broken run's loader was sequential (it supersedes the `RandomSampler` reading logged on 2026-10-21). The train file is stored positives-first, so batches were long runs of one label. Consistent with the queue: `mean` / seed 0 (same seed and pooling, shuffled loader) trained normally from epoch 0 (found 78.7, val loss 0.024, the same as the first good run).

## 4. Single test run
Only after §1–2 are decided, and only once: `load_pairs("overture", "test")`, a non-shuffled loader, `rebuild_model` on the chosen checkpoint, `evaluate`. Then the `results.md` row + paragraph.

In [6]:
pooling_means = summary.groupby(["pooling", "seed"])[["train_loss", "val_loss", "found", "case1", "easy"]].mean()
display(pooling_means)
def _find_seed_noise(label):
    max_value = pooling_means.loc["mean", label].max()
    min_value = pooling_means.loc["mean", label].min()
    return max_value-min_value

for item in ["train_loss", "val_loss", "found","case1", "easy"]:
    print (f"diff in seed noise for column: {item}")
    print(_find_seed_noise(item))


train_loss  val_loss      found      case1       easy
pooling seed                                                       
max     0       0.004218  0.018681  88.958595  99.274363  99.895235
        1       0.007373  0.015520  88.205772  99.317901  99.886854
        2       0.008196  0.018080  90.213300  98.955083  99.832376
mean    0       0.003717  0.016915  87.076537  99.434003  99.932951
        1       0.010259  0.014307  87.076537  99.499311  99.899426
        2       0.008865  0.015387  90.840652  99.201800  99.878473

diff in seed noise for column: train_loss
0.006542080898575765
diff in seed noise for column: val_loss
0.0026076827198266983
diff in seed noise for column: found
3.7641154328732824
diff in seed noise for column: case1
0.2975110659603786
diff in seed noise for column: easy
0.05447764321334603


In [7]:
ot_test, ot_test_label = load_pairs("overture", "test")
test_loader = DataLoader(PairDataset(ot_test, ot_test_label), batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(vocab))   

ckpt = torch.load("checkpoints/overture_siamese_seed_0_pooling_mean.pt")

model = rebuild_model(ckpt)
# res = evaluate(model, test_loader, ot_test, ot_test_label)
# print(res)
val_preds = (predict(model, val_loader) >= THRESHOLD).int().numpy()
val_slices = review_typical_slices(ot_val, val_preds, ot_val_label)
print(val_slices)
preds = (predict(model, test_loader) >= THRESHOLD).int().numpy()
test_slices = review_typical_slices(ot_test, preds, ot_test_label)
print(test_slices)

                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3390       0    100.0%
1                  positives: non-identical    797      694     103     87.1%
2     neg: case1 (same street, diff number)  13781    13703      78     99.4%
3  neg: case2 (same number, similar street)     34       10      24     29.4%
4                         neg: case3 (easy)  22969    22953      16     99.9%
5   neg: case2s (swapped look-alike street)    894      222     672     24.8%
6                        overall (accuracy)  41865    40972     893     97.9%


                                      slice      n  correct  errors % correct
0                      positives: identical   3543     3543       0    100.0%
1                  positives: non-identical    843      705     138     83.6%
2     neg: case1 (same street, diff number)  14448    14367      81     99.4%
3  neg: case2 (same number, similar street)     15        5      10     33.3%
4                         neg: case3 (easy)  24995    24972      23     99.9%
5                        overall (accuracy)  43844    43592     252     99.4%


In [8]:
# Siamese trained on case2s, mean vs max pooling (seed 0): % correct per slice, "test (val)", + AUC non-identical vs case2s
# (mean reproduces its results.md row; max is the 2026-11-18 run). Feeds the comparison table below.
from sklearn.metrics import roc_auc_score

c2_splits = {split: load_pairs("overture_case2s", split) for split in ["val", "test"]}
siamese_c2 = {}   # (pooling, split) -> (preds, probs)
for pooling in ["mean", "max"]:
    _, ck = load_run(0, pooling, checkpoint_dir="checkpoints_case2s")
    m = rebuild_model(ck)
    for split, (df, lab) in c2_splits.items():
        loader = DataLoader(PairDataset(df, lab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(ck["vocab"]))
        probs = predict(m, loader).numpy()
        siamese_c2[(pooling, split)] = ((probs >= THRESHOLD).astype(int), probs)

def _auc(df, lab, probs):
    lab = np.asarray(lab).ravel()
    mask = (((df["neg_type"] == "case0") & (df["address_a"] != df["address_b"])) | (df["neg_type"] == "case2s")).values
    return roc_auc_score(lab[mask], probs[mask])

rows = {}
for pooling in ["mean", "max"]:
    cols = {}
    for split, (df, lab) in c2_splits.items():
        preds, probs = siamese_c2[(pooling, split)]
        cols[split] = review_typical_slices(df, preds, lab).set_index("slice")["% correct"]
        cols[split]["AUC, non-identical vs case2s"] = f"{_auc(df, lab, probs):.3f}"
    rows[f"Siamese case2s, {pooling} pooling"] = cols["test"].str.rstrip("%") + " (" + cols["val"].str.rstrip("%") + ")"
pd.DataFrame(rows).T

slice,positives: identical,positives: non-identical,"neg: case1 (same street, diff number)","neg: case2 (same number, similar street)",neg: case3 (easy),neg: case2s (swapped look-alike street),overall (accuracy),"AUC, non-identical vs case2s"
"Siamese case2s, mean pooling",100.0 (100.0),80.1 (83.1),99.3 (99.3),53.3 (64.7),100.0 (100.0),75.7 (74.5),98.8 (98.8),0.854 (0.871)
"Siamese case2s, max pooling",100.0 (100.0),78.3 (79.3),99.0 (99.2),33.3 (64.7),100.0 (100.0),76.5 (76.6),98.6 (98.8),0.835 (0.847)


**Comparison on the `case2s` dataset** (copied from `results.md` → *Overture `case2s`*, except the max-pooling siamese row, from the scoring cell above; every model trained on the `case2s` train split; test, val in brackets; % found for matches, % rejected for non-matches; libpostal left out)

| Model | Step | Identical matches — **% found** | **Non-identical matches — % found** | Case1 non-matches — **% rejected** | **Case2s non-matches — % rejected** | Easy non-matches — **% rejected** | **AUC, non-identical vs case2s** |
|---|---|---|---|---|---|---|---|
| LogReg (3 Jaro-Winkler features) | 1 | 99.9 (100.0) | **59.2** (62.4) | 99.7 (99.5) | **59.6** (63.4) | 100.0 (100.0) | 0.568 (0.622) |
| LogReg (23 similarity features) | 1b | 99.9 (100.0) | **50.5** (57.1) | 99.1 (99.2) | **74.4** (77.0) | 100.0 (100.0) | 0.687 (0.724) |
| Random forest | 1b | 100.0 (100.0) | **70.1** (76.8) | 99.6 (99.5) | **88.7** (91.3) | 99.9 (99.9) | 0.902 (0.933) |
| XGBoost | 1b | 100.0 (100.0) | **69.9** (76.3) | 99.6 (99.5) | **88.5** (90.2) | 100.0 (100.0) | 0.902 (0.931) |
| Siamese BiLSTM, trained on `case2s` (mean pooling, seed 0) | 3 | 100.0 (100.0) | **80.1** (83.1) | 99.3 (99.3) | **75.7** (74.5) | 100.0 (100.0) | 0.854 (0.871) |
| Siamese BiLSTM, trained on `case2s` (max pooling, seed 0) | 3 | 100.0 (100.0) | **78.3** (79.3) | 99.0 (99.2) | **76.5** (76.6) | 100.0 (100.0) | 0.835 (0.847) |
| ESIM, word2vec `min_count=2` (reproduction; 3 seeds, test min–max) | 4 | 95.5–100.0 (97.9–100.0) | **77.9–80.7** (80.9–82.7) | 96.1–98.4 (98.2–99.1) | **73.9–75.5** (76.1–80.0) | 99.9 (99.9–100.0) | 0.845–0.850 (0.867–0.883) |
| ESIM, word2vec `min_count=2` + random row per rare token (3 seeds, test min–max) | 4 | 99.9–100.0 (100.0) | **74.9–78.9** (77.8–81.8) | 98.6–98.9 (99.0–99.2) | **72.9–75.9** (74.6–82.2) | 99.9–100.0 (100.0) | 0.815–0.842 (0.856–0.878) |
| *test n (pairs in the slice)* | | *3,543* | *843* | *14,448* | *1,053* | *23,942* | *843 vs 1,053* |

**By house-number group, per `neg_type`** (case2s **test**; groups from `add_number_street`: exactly one address has no number / both have a number and they differ; one table per group). % correct: `case0` = matches found, the rest = non-matches rejected. XGBoost refit on the cached case2s features (same settings as 1b, deterministic); siamese = case2s checkpoint (mean pooling, seed 0); ESIM = saved test predictions, min–max over 3 seeds. Types with no such pairs are left out; case2 left out (too few rows, as in the table above).

In [9]:
# By house-number group (one side missing / both present, different), case2s test: % correct per neg_type, per model
from xgboost import XGBClassifier
from helpers.pairs import add_number_street
from helpers.features import get_processed_dfs
from esim.training import SETTINGS as ESIM_SETTINGS

c2 = {split: load_pairs("overture_case2s", split) for split in ["train", "val", "test"]}
c2_test = add_number_street(c2["test"][0])
c2_test_label = c2["test"][1]

# XGBoost: refit on the cached case2s features (model_baseline/07_backtest_case2s.ipynb's cache)
c2_X = get_processed_dfs([[add_number_street(c2[split][0]), f"../model_baseline/case2s_training/case2s_evaluation_{split}.parquet"]
                          for split in ["train", "val", "test"]])
xgb_c2 = XGBClassifier(random_state=0).fit(c2_X[0], c2["train"][1])

# Siamese: case2s checkpoint, predicted on case2s test with its own vocab
history_c2, ckpt_c2 = load_run(0, "mean", checkpoint_dir="checkpoints_case2s")
c2_test_loader = DataLoader(PairDataset(c2["test"][0], c2_test_label), batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=make_collate(ckpt_c2["vocab"]))

model_preds = {
    "XGBoost [1b]": [xgb_c2.predict(c2_X[2])],
    "Siamese BiLSTM, case2s (mean, seed 0) [3]": [(predict(rebuild_model(ckpt_c2), c2_test_loader) >= THRESHOLD).int().numpy()],
    "Siamese BiLSTM, case2s (max, seed 0) [3]": [siamese_c2[("max", "test")][0]],
    "ESIM mc2 (3 seeds) [4]": [np.load(f"{ESIM_SETTINGS['mc2']}/preds_test_seed_{s}.npy") for s in SEEDS],
    "ESIM rand_mc2 (3 seeds) [4]": [np.load(f"{ESIM_SETTINGS['rand_mc2']}/preds_test_seed_{s}.npy") for s in SEEDS],
}

has_a, has_b = c2_test["number_a"].notna(), c2_test["number_b"].notna()
number_groups = {
    "one side missing": has_a != has_b,
    "both present, different": has_a & has_b & (c2_test["number_a"] != c2_test["number_b"]),
}
labels_s = pd.Series(np.asarray(c2_test_label).ravel(), index=c2_test.index)

def pct_range(preds_list, mask):
    vals = [((pd.Series(np.asarray(p).ravel(), index=c2_test.index) == labels_s)[mask]).mean() * 100 for p in preds_list]
    return f"{min(vals):.1f}" if len(vals) == 1 or min(vals) == max(vals) else f"{min(vals):.1f}–{max(vals):.1f}"

for group, group_mask in number_groups.items():
    # case2 left out (too few rows); types with no pairs in this group are skipped
    neg_types = [t for t in ["case0", "case1", "case2s", "case3"] if (group_mask & (c2_test["neg_type"] == t)).any()]
    table = pd.DataFrame({
        name: {t: pct_range(preds_list, group_mask & (c2_test["neg_type"] == t)) for t in neg_types}
        for name, preds_list in model_preds.items()
    }).T
    table.loc["test n"] = {t: int((group_mask & (c2_test["neg_type"] == t)).sum()) for t in neg_types}
    print(f"{'='*20} {group} {'='*20}")
    display(table)

/Users/david/Downloads/ml-geocoding/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


==================== one side missing ====================


,case0,case1,case3
XGBoost [1b],22.6,84.9,99.3
"Siamese BiLSTM, case2s (mean, seed 0) [3]",32.9,71.2,99.3
"Siamese BiLSTM, case2s (max, seed 0) [3]",45.7,59.0,99.0
ESIM mc2 (3 seeds) [4],19.5–28.7,91.3–98.3,99.2–99.9
ESIM rand_mc2 (3 seeds) [4],12.8–29.9,90.4–100.0,99.4–99.9
test n,164,344,990


==================== both present, different ====================


,case0,case1,case3
XGBoost [1b],51.2,100.0,100.0
"Siamese BiLSTM, case2s (mean, seed 0) [3]",95.1,100.0,100.0
"Siamese BiLSTM, case2s (max, seed 0) [3]",95.1,100.0,100.0
ESIM mc2 (3 seeds) [4],92.7–97.6,96.0–98.5,99.9–100.0
ESIM rand_mc2 (3 seeds) [4],95.1–97.6,98.7–99.1,100.0–100.0
test n,41,14104,22884


In [10]:
# ROC AUC within "one side missing" (case2s test): threshold-free, so it separates ranking from where 0.5 sits.
# Positives = case0 in the group; negatives = case1 in the group (the trade-off column), or every negative in the group.
probs = {
    "XGBoost [1b]": xgb_c2.predict_proba(c2_X[2])[:, 1],
    "Siamese BiLSTM, case2s (mean, seed 0) [3]": siamese_c2[("mean", "test")][1],
    "Siamese BiLSTM, case2s (max, seed 0) [3]": siamese_c2[("max", "test")][1],
}
one_missing = (has_a != has_b).values
nt = c2_test["neg_type"].values
lab = labels_s.values
comparisons = {
    "case0 vs case1": one_missing & np.isin(nt, ["case0", "case1"]),
    "case0 vs all negatives": one_missing,
}
auc_missing = pd.DataFrame({
    name: {comp: round(roc_auc_score(lab[mask], p[mask]), 3) for comp, mask in comparisons.items()}
    for name, p in probs.items()
}).T
auc_missing.loc["n (pos / neg)"] = {comp: f"{int(lab[mask].sum())} / {int((lab[mask] == 0).sum())}" for comp, mask in comparisons.items()}
auc_missing

,case0 vs case1,case0 vs all negatives
XGBoost [1b],0.537,0.868
"Siamese BiLSTM, case2s (mean, seed 0) [3]",0.526,0.764
"Siamese BiLSTM, case2s (max, seed 0) [3]",0.545,0.784
n (pos / neg),164 / 344,164 / 1335


In [11]:
# case2s test pairs: XGBoost right, siamese (case2s, mean, seed 0) wrong
xgb_right = pd.Series(model_preds["XGBoost [1b]"][0], index=c2_test.index) == labels_s
siam_right = pd.Series(model_preds["Siamese BiLSTM, case2s (mean, seed 0) [3]"][0], index=c2_test.index) == labels_s
gap = (c2_test["neg_type"] == "case2s") & xgb_right & ~siam_right

print(gap.sum(), "of", (c2_test["neg_type"] == "case2s").sum(), "case2s pairs")
c2_test[gap][["address_a", "address_b", "street_a", "street_b"]].sample(20, random_state=0)


180 of 1053 case2s pairs


,address_a,address_b,street_a,street_b
43447,372 SCOTT SWAMP RD,372 SCOTT SWAP RD,SCOTT SWAMP RD,SCOTT SWAP RD
42857,930 STRAITS TURNPIKE,930 STRAITS TOKE,STRAITS TURNPIKE,STRAITS TOKE
43558,2321 MERIDEN WATERBURY TPKE,2321 MERIDEN-WTBY,MERIDEN WATERBURY TPKE,MERIDEN-WTBY
43731,36 SHUNPIKE RD ROUTE 3,36 SHUNPIKE RD STE 3,SHUNPIKE RD ROUTE 3,SHUNPIKE RD STE 3
43606,30 CREST RD,30 CHRISTIAN RD,CREST RD,CHRISTIAN RD
43268,167 WATERFORD PKWY N,167 WATERFORD PARKWAY SOUTH,WATERFORD PKWY N,WATERFORD PARKWAY SOUTH
43185,88 SCOTT SWAMP RD,88 SCOTT SWAP RD,SCOTT SWAMP RD,SCOTT SWAP RD
43683,1 ANNGINA DR,1 ANNGINNA DR,ANNGINA DR,ANNGINNA DR
43751,520 HARTFORD TURNPIKE K,520 HARTFORD TRPK,HARTFORD TURNPIKE K,HARTFORD TRPK
43450,225 WEST ST,225 WEST ST ROUTE 67,WEST ST,WEST ST ROUTE 67


## Diagnostics
One-off queries and their results (question → result → query). Kept out of the analysis cells.